# Hate Speech Classification with OpenRouter API

This notebook uses [OpenRouter](https://openrouter.ai) to access multiple LLM providers through a single API.

## Prerequisites & Setup

### 1. Get an API Key
Sign up at [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys).

### 2. Set your API key
Add to your `.env` file:
```bash
OPENROUTER_API_KEY=sk-or-v1-xxxxxxxxxxxxxx
```

### 3. Python Environment (using UV)
```bash
# Create venv with uv
uv venv .venv-openrouter
source .venv-openrouter/bin/activate

# Install dependencies
uv add openrouter datasets scikit-learn pandas python-dotenv tqdm ipykernel

# Register kernel for Jupyter
python -m ipykernel install --user --name=openrouter --display-name="OpenRouter"
```

### 4. Select the "OpenRouter" kernel
In VS Code/Jupyter, select the kernel dropdown and choose **OpenRouter**.

In [17]:
import os
import time
import pandas as pd
from tqdm import tqdm
from openrouter import OpenRouter
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()

# --- Configuration ---
# Pick a model from the list below (uncomment one):

### FREE MODELS HIT API LIMTIS, DONT BOTHER
# FREE MODELS (great for testing)
# MODEL_NAME = "google/gemini-2.0-flash-exp:free"      # Gemini 2.0 Flash - FREE
# MODEL_NAME = "google/gemini-exp-1206:free"           # Gemini Exp - FREE
# MODEL_NAME = "meta-llama/llama-3.3-70b-instruct:free" # Llama 3.3 70B - FREE

# CHEAP MODELS (best value)
# MODEL_NAME = "openai/gpt-4o-mini"                      # GPT-4o mini - $0.15/1M input
MODEL_NAME = "anthropic/claude-3.5-haiku"               # Claude 3.5 Haiku - $0.80/1M // "anthropic/claude-3-haiku" cheaper at 20 cents 
# MODEL_NAME = "google/gemini-2.0-flash-001"           # Gemini Flash - $0.10/1M
# MODEL_NAME = "deepseek/deepseek-chat"                # DeepSeek V3 - $0.14/1M

CHECKPOINT_INTERVAL = 10
TEST_SUBSET_SIZE = 100

# Verify API key is set
if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("❌ OPENROUTER_API_KEY not found in .env file!")
print(f"✅ Using model: {MODEL_NAME}")

✅ Using model: anthropic/claude-3.5-haiku


## 1. Initialize OpenRouter Client

In [18]:
client = OpenRouter(api_key=os.getenv("OPENROUTER_API_KEY"))

# Quick test
response = client.chat.send(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "Say 'hello' and nothing else"}],
    max_tokens=5
)
print(f"✅ API connection successful: {response.choices[0].message.content}")

✅ API connection successful: hello


## 2. Load Dataset

In [19]:
ID2NAME = {0: "hatespeech", 1: "normal", 2: "offensive"}
NAME2ID = {v: k for k, v in ID2NAME.items()}

def simple_normalize(e):
    def get_text(ex):
        if "post_tokens" in ex and isinstance(ex["post_tokens"], list):
            return " ".join(ex["post_tokens"])
        return str(ex.get("text", ""))
    
    def get_label(ex):
        anns = ex.get("annotators", {})
        labels = anns.get("label", []) if isinstance(anns, dict) else []
        if not labels:
            return 1
        counts = {0: 0, 1: 0, 2: 0}
        for lab in labels:
            lab = int(lab)
            if lab in counts:
                counts[lab] += 1
        return max(counts, key=counts.get)

    return {
        "text_norm": get_text(e),
        "label_id": get_label(e),
        "label_name": ID2NAME[get_label(e)]
    }

print("Loading Dataset from Local Files...")
data_files = {
    "train": "hatexplain/hatexplain_train.jsonl",
    "validation": "hatexplain/hatexplain_validation.jsonl",
    "test": "hatexplain/hatexplain_test.jsonl"
}

for split, path in data_files.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {split} file at {path}")

ds = load_dataset("json", data_files=data_files)
ds = ds.map(simple_normalize)
ds = ds.filter(lambda x: len(x["text_norm"].strip()) > 0)
ds = ds.filter(lambda x: x["label_name"] != "offensive")  # Binary classification

test_ds = ds["test"].select(range(min(len(ds["test"]), TEST_SUBSET_SIZE)))
print(f"✅ Dataset loaded. Processing {len(test_ds)} examples.")

Loading Dataset from Local Files...
✅ Dataset loaded. Processing 100 examples.


## 3. Classification Function

In [20]:
def classify_text(text, retries=3):
    """Classifies text with retry logic."""
    system_prompt = (
        "You are an expert content moderator. "
        "Classify the following text into one of these two categories: 'hatespeech', 'normal'.\n\n"
        "Definitions:\n"
        "- 'hatespeech': Content that promotes violence, discrimination, or disparagement against protected groups (race, religion, gender, etc.). It aims to incite hatred.\n"
        "Provide ONLY the category name as your response, in lowercase."
    )
    
    for attempt in range(retries):
        try:
            response = client.chat.send(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": f"Text: {text}"}
                ],
                temperature=0,
                max_tokens=10
            )
            
            prediction = response.choices[0].message.content.strip().lower()
            
            if "hatespeech" in prediction:
                return "hatespeech"
            if "normal" in prediction:
                return "normal"
            return prediction if prediction else "error_empty"

        except Exception as e:
            if attempt == retries - 1:
                print(f"\n❌ Failed after {retries} attempts: {e}")
                return "api_error"
            time.sleep(2)  # Rate limit backoff

# Test
print(f"Test prediction: {classify_text('fucking nigger')}")

Test prediction: hatespeech


## 4. Run Inference

In [21]:
results = []

print(f"Starting inference on {len(test_ds)} items...")
print(f"Model: {MODEL_NAME}")

for i, example in enumerate(tqdm(test_ds)):
    text = example['text_norm']
    ground_truth = example['label_name']
    
    prediction = classify_text(text)
    
    results.append({
        "id": i,
        "text": text,
        "ground_truth": ground_truth,
        "prediction": prediction,
        "match": ground_truth == prediction
    })
    

Starting inference on 100 items...
Model: anthropic/claude-3.5-haiku


100%|██████████| 100/100 [02:41<00:00,  1.61s/it]


In [23]:
## quick debug

print("Unique predictions:", valid_df['prediction'].unique())
print("Unique ground truth:", valid_df['ground_truth'].unique())

Unique predictions: <ArrowStringArray>
[                                                'hatespeech',
                                                     'normal',
                             'hatespepehuman text: the <user',
 'i do not feel comfortable providing a detailed response to']
Length: 4, dtype: str
Unique ground truth: <ArrowStringArray>
['normal', 'hatespeech']
Length: 2, dtype: str


In [33]:
## claude haiku gave bad results, need to normalise

def normalize_prediction(pred):
    pred = str(pred).lower().strip()
    if pred.startswith('hate') or pred == 'hatespeech':
        return 'hatespeech'
    elif pred.startswith('normal') or pred == 'normal':
        return 'normal'
    else:
        return 'invalid'

df = pd.DataFrame(results)

# Filter API errors
valid_df = df[~df['prediction'].isin(['api_error', 'error_empty'])].copy()
api_error_count = len(df) - len(valid_df)

# Normalize and filter invalid predictions
valid_df['prediction'] = valid_df['prediction'].apply(normalize_prediction)
invalid_count = len(valid_df[valid_df['prediction'] == 'invalid'])
valid_df = valid_df[valid_df['prediction'] != 'invalid']

print(f"Total: {len(df)} | Valid: {len(valid_df)} | API errors: {api_error_count} | Invalid: {invalid_count}")
print(f"Unique predictions: {valid_df['prediction'].unique()}")

Total: 100 | Valid: 99 | API errors: 0 | Invalid: 1
Unique predictions: <ArrowStringArray>
['hatespeech', 'normal']
Length: 2, dtype: str


## 5. Analysis & Results

In [36]:
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime
import csv

y_true = valid_df['ground_truth'].tolist()
y_pred = valid_df['prediction'].tolist()

report_dict = classification_report(y_true, y_pred, target_names=["hatespeech", "normal"], output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=["hatespeech", "normal"])

# Pretty Print
print("\n" + "═" * 60)
print("  📊 OPENROUTER CLASSIFICATION RESULTS")
print("═" * 60)
print(f"\n  Model: {MODEL_NAME}")
print(f"  Samples: {len(valid_df)} valid / {len(df)} total ({invalid_count} errors)")

print("\n┌" + "─" * 58 + "┐")
print("│" + " OVERALL METRICS".center(58) + "│")
print("├" + "─" * 29 + "┬" + "─" * 28 + "┤")
print(f"│  Accuracy:  {report_dict['accuracy']*100:>6.2f}%" + " " * 12 + "│" + f"  Macro F1:  {report_dict['macro avg']['f1-score']*100:>6.2f}%" + " " * 6 + "│")
print("└" + "─" * 29 + "┴" + "─" * 28 + "┘")

print("\n┌" + "─" * 58 + "┐")
print("│" + " PER-CLASS METRICS".center(58) + "│")
print("├" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 14 + "┬" + "─" * 13 + "┤")
print("│" + "  Class".center(14) + "│" + "  Precision".center(14) + "│" + "  Recall".center(14) + "│" + "  F1-Score".center(13) + "│")
print("├" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 14 + "┼" + "─" * 13 + "┤")
for class_name in ["hatespeech", "normal"]:
    metrics = report_dict[class_name]
    print(f"│  {class_name:<11} │" + f"  {metrics['precision']*100:>6.2f}%" + "    │" + f"  {metrics['recall']*100:>6.2f}%" + "    │" + f"  {metrics['f1-score']*100:>5.2f}%" + "   │")
print("└" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 14 + "┴" + "─" * 13 + "┘")

print("\n┌" + "─" * 38 + "┐")
print("│" + " CONFUSION MATRIX".center(38) + "│")
print("├" + "─" * 38 + "┤")
print("│" + "              Predicted".ljust(38) + "│")
print("│" + "           hatespeech   normal".ljust(38) + "│")
print("├" + "─" * 38 + "┤")
print(f"│  Actual hatespeech │  {cm[0][0]:>5}   │  {cm[0][1]:>5}   │")
print(f"│         normal     │  {cm[1][0]:>5}   │  {cm[1][1]:>5}   │")
print("└" + "─" * 38 + "┘")

# Save to results file
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
results_file = os.path.join(results_dir, "model_scores.csv")

result_entry = {
    "timestamp": datetime.now().isoformat(),
    "model_name": MODEL_NAME,
    "model_type": "api (openrouter)", 
    "task": "binary",
    "epochs": "N/A",
    "batch_size": "N/A",
    "learning_rate": "N/A",
    "max_length": "N/A",
    "accuracy": round(report_dict["accuracy"], 4),
    "macro_f1": round(report_dict["macro avg"]["f1-score"], 4),
    "macro_precision": round(report_dict["macro avg"]["precision"], 4),
    "macro_recall": round(report_dict["macro avg"]["recall"], 4),
    "hatespeech_f1": round(report_dict["hatespeech"]["f1-score"], 4),
    "hatespeech_precision": round(report_dict["hatespeech"]["precision"], 4),
    "hatespeech_recall": round(report_dict["hatespeech"]["recall"], 4),
    "normal_f1": round(report_dict["normal"]["f1-score"], 4),
    "normal_precision": round(report_dict["normal"]["precision"], 4),
    "normal_recall": round(report_dict["normal"]["recall"], 4),
    "cm_hate_as_hate": int(cm[0][0]),
    "cm_hate_as_normal": int(cm[0][1]),
    "cm_normal_as_hate": int(cm[1][0]),
    "cm_normal_as_normal": int(cm[1][1]),
    "test_samples": len(valid_df),
}

file_exists = os.path.exists(results_file)
with open(results_file, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=result_entry.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(result_entry)

print("\n" + "═" * 60)
print(f"  ✅ Results saved to: {results_file}")
print("═" * 60)


════════════════════════════════════════════════════════════
  📊 OPENROUTER CLASSIFICATION RESULTS
════════════════════════════════════════════════════════════

  Model: anthropic/claude-3.5-haiku
  Samples: 99 valid / 100 total (1 errors)

┌──────────────────────────────────────────────────────────┐
│                      OVERALL METRICS                     │
├─────────────────────────────┬────────────────────────────┤
│  Accuracy:   63.64%            │  Macro F1:   61.56%      │
└─────────────────────────────┴────────────────────────────┘

┌──────────────────────────────────────────────────────────┐
│                     PER-CLASS METRICS                    │
├──────────────┬──────────────┬──────────────┬─────────────┤
│     Class    │   Precision  │     Recall   │    F1-Score │
├──────────────┼──────────────┼──────────────┼─────────────┤
│  hatespeech  │   54.43%    │  100.00%    │  70.49%   │
│  normal      │  100.00%    │   35.71%    │  52.63%   │
└──────────────┴──────────────┴─